In [1]:
import os
import glob
import random
import xml.etree.ElementTree as ET
import shutil

In [2]:
# ----------------------------------------------------------
# 1) 사용자 지정 파라미터
# ----------------------------------------------------------
# (1) XML + 이미지가 섞여 있는 폴더 경로 (원본)
DATA_DIR = "/mnt/d/ships/ships"

# (2) 결과물을 저장할 폴더 (images/train, images/val, labels/train, labels/val)
OUTPUT_DIR = "/mnt/d/ships/datasets"

# (3) 사용할 클래스 목록 (예시)
CLASSES = ["fising_boat", "boat", "buoy"]
# YOLO에서는 0부터 클래스 ID를 매기므로, 위 순서대로 0,1,2,...

# (4) train/val 분할 비율
SPLIT_RATIO = 0.8  # 80%는 train, 20%는 val

# (5) 이미지 확장자
#   - 보통 .jpg, .png 등이 섞여 있을 수 있으므로
#   - 아래 처리를 참고하여 필요하면 바꾸세요.
IMAGE_EXT = [".jpg", ".jpeg", ".png"]

In [3]:
os.makedirs(os.path.join(OUTPUT_DIR, "images/train"), exist_ok=True)
os.makedirs(os.path.join(OUTPUT_DIR, "images/val"), exist_ok=True)
os.makedirs(os.path.join(OUTPUT_DIR, "labels/train"), exist_ok=True)
os.makedirs(os.path.join(OUTPUT_DIR, "labels/val"), exist_ok=True)

In [4]:
def convert_voc_to_yolo(size, box):
    """
    size: (w, h) = 이미지의 폭, 높이
    box: (xmin, ymin, xmax, ymax)
    결과: (x_center, y_center, width, height) [0~1 정규화]
    """
    dw = 1.0 / size[0]
    dh = 1.0 / size[1]
    x_center = (box[0] + box[2]) / 2.0
    y_center = (box[1] + box[3]) / 2.0
    w = box[2] - box[0]
    h = box[3] - box[1]
    x_center *= dw
    w *= dw
    y_center *= dh
    h *= dh
    return (x_center, y_center, w, h)

In [5]:
def parse_xml(xml_path):
    """
    xml_path 파일에서
    - (width, height)
    - objects (class_id, [YOLO x_center, y_center, w, h])
    리스트를 추출하여 반환
    """
    tree = ET.parse(xml_path)
    root = tree.getroot()

    size = root.find("size")
    width = int(size.find("width").text)
    height = int(size.find("height").text)

    bboxes = []
    for obj in root.iter("object"):
        cls_name = obj.find("name").text.strip()
        if cls_name not in CLASSES:
            # CLASSES에 없는 객체는 건너뛰거나 로깅
            continue
        class_id = CLASSES.index(cls_name)

        bndbox = obj.find("bndbox")
        xmin = float(bndbox.find("xmin").text)
        ymin = float(bndbox.find("ymin").text)
        xmax = float(bndbox.find("xmax").text)
        ymax = float(bndbox.find("ymax").text)

        x_c, y_c, w, h = convert_voc_to_yolo((width, height), (xmin, ymin, xmax, ymax))
        bboxes.append((class_id, x_c, y_c, w, h))

    return width, height, bboxes

In [6]:
xml_files = glob.glob(os.path.join(DATA_DIR, "*.xml"))

dataset = []
for xml_file in xml_files:
    # base_name: 예) "boat_b_7_0000001"
    base_name = os.path.splitext(os.path.basename(xml_file))[0]

    # xml에 해당하는 이미지 파일 찾기
    #   - base_name + (jpg/png/...) 형태
    #   - 실제 존재하는 확장자를 확인
    matched_img_path = None
    for ext in IMAGE_EXT:
        candidate = os.path.join(DATA_DIR, base_name + ext)
        if os.path.exists(candidate):
            matched_img_path = candidate
            break

    if matched_img_path is None:
        # 이미지가 존재하지 않으면 스킵
        continue

    # xml 파싱하여 바운딩 박스 정보 얻기
    _, _, bboxes = parse_xml(xml_file)

    # 바운딩박스가 하나도 없다면 굳이 dataset에 넣지 않아도 됨
    if len(bboxes) == 0:
        continue

    # YOLO 라벨 문자열 만들기
    # class_id x_center y_center w h (각각 0~1 범위)
    label_str_lines = []
    for bbox in bboxes:
        cid, x_c, y_c, w, h = bbox
        label_str_lines.append(f"{cid} {x_c:.6f} {y_c:.6f} {w:.6f} {h:.6f}")

    label_str = "\n".join(label_str_lines)
    dataset.append((matched_img_path, label_str))

print(f"총 {len(dataset)} 세트가 준비되었습니다.")


총 19219 세트가 준비되었습니다.


In [7]:
random.shuffle(dataset)
train_size = int(len(dataset) * SPLIT_RATIO)
train_data = dataset[:train_size]
val_data = dataset[train_size:]

print(f"→ train: {len(train_data)}, val: {len(val_data)}")

# ----------------------------------------------------------
# 6) 이미지/라벨을 실제 폴더에 복사
# ----------------------------------------------------------
def save_data(data_split, subset_name):
    """
    data_split: (img_path, label_str) 리스트
    subset_name: "train" 또는 "val"
    """
    for img_path, label_str in data_split:
        # 6-1) 이미지 복사
        dst_img_path = os.path.join(OUTPUT_DIR, "images", subset_name, os.path.basename(img_path))
        shutil.copy2(img_path, dst_img_path)

        # 6-2) 라벨 txt 생성
        txt_name = os.path.splitext(os.path.basename(img_path))[0] + ".txt"
        dst_label_path = os.path.join(OUTPUT_DIR, "labels", subset_name, txt_name)
        with open(dst_label_path, "w", encoding="utf-8") as f:
            f.write(label_str)

save_data(train_data, "train")
save_data(val_data, "val")
print("완료! 데이터셋이 생성되었습니다.")

→ train: 15375, val: 3844
완료! 데이터셋이 생성되었습니다.


In [9]:
from ultralytics import YOLO

# 1) 모델 로드 (사전학습된 YOLOv8)
model = YOLO('yolov8n.pt')  
#   - 만약 커스텀 프리트레인 가중치가 있다면, 'runs/detect/train/weights/best.pt'처럼 직접 경로 지정

# 2) 모델 학습(Training)
model.train(
    data='/mnt/d/ships/datasets/dataset.yaml',  # dataset.yaml 경로
    epochs=50,            # 원하는 만큼 (예: 50, 100...)
    imgsz=640,            # 이미지 사이즈 (기본 640)
    batch=16,             # 배치 사이즈 (원하는 값으로 조정)
    name='/mnt/d/ships/model' # 결과 저장 폴더 이름 (runs/detect/my_ships_model/)
)

Ultralytics 8.3.58 🚀 Python-3.10.15 torch-2.5.0 CUDA:0 (NVIDIA GeForce RTX 4090, 24564MiB)
engine/trainer: task=detect, mode=train, model=yolov8n.pt, data=/mnt/d/ships/datasets/dataset.yaml, epochs=50, time=None, patience=100, batch=16, imgsz=640, save=True, save_period=-1, cache=False, device=None, workers=8, project=None, name=model2, exist_ok=False, pretrained=True, optimizer=auto, verbose=True, seed=0, deterministic=True, single_cls=False, rect=False, cos_lr=False, close_mosaic=10, resume=False, amp=True, fraction=1.0, profile=False, freeze=None, multi_scale=False, overlap_mask=True, mask_ratio=4, dropout=0.0, val=True, split=val, save_json=False, save_hybrid=False, conf=None, iou=0.7, max_det=300, half=False, dnn=False, plots=True, source=None, vid_stride=1, stream_buffer=False, visualize=False, augment=False, agnostic_nms=False, classes=None, retina_masks=False, embed=None, show=False, save_frames=False, save_txt=False, save_conf=False, save_crop=False, show_labels=True, show_con

100%|██████████| 5.35M/5.35M [00:00<00:00, 11.6MB/s]


AMP: checks passed ✅


train: Scanning /mnt/d/ships/datasets/labels/train... 15375 images, 0 backgrounds, 0 corrupt: 100%|██████████| 15375/15375 [00:19<00:00, 787.81it/s]


train: New cache created: /mnt/d/ships/datasets/labels/train.cache


...

                   all       3844       4522      0.962       0.96      0.981      0.793



50 epochs completed in 1.037 hours.
Optimizer stripped from /mnt/d/ships/model2/weights/last.pt, 6.2MB
Optimizer stripped from /mnt/d/ships/model2/weights/best.pt, 6.2MB

Validating /mnt/d/ships/model2/weights/best.pt...
Ultralytics 8.3.58 🚀 Python-3.10.15 torch-2.5.0 CUDA:0 (NVIDIA GeForce RTX 4090, 24564MiB)
Model summary (fused): 168 layers, 3,006,233 parameters, 0 gradients, 8.1 GFLOPs


                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100%|██████████| 121/121 [00:11<00:00, 10.69it/s]


                   all       3844       4522      0.964      0.961       0.98      0.793
                  boat       3365       3642       0.98      0.993      0.994      0.844
                  buoy        716        880      0.949      0.928      0.966      0.743
Speed: 0.1ms preprocess, 0.3ms inference, 0.0ms loss, 0.5ms postprocess per image
Results saved to /mnt/d/ships/model2


ultralytics.utils.metrics.DetMetrics object with attributes:

ap_class_index: array([1, 2])
box: ultralytics.utils.metrics.Metric object
confusion_matrix: <ultralytics.utils.metrics.ConfusionMatrix object at 0x7fb505db63b0>
curves: ['Precision-Recall(B)', 'F1-Confidence(B)', 'Precision-Confidence(B)', 'Recall-Confidence(B)']
curves_results: [[array([          0,    0.001001,    0.002002,    0.003003,    0.004004,    0.005005,    0.006006,    0.007007,    0.008008,    0.009009,     0.01001,    0.011011,    0.012012,    0.013013,    0.014014,    0.015015,    0.016016,    0.017017,    0.018018,    0.019019,     0.02002,    0.021021,    0.022022,    0.023023,
          0.024024,    0.025025,    0.026026,    0.027027,    0.028028,    0.029029,     0.03003,    0.031031,    0.032032,    0.033033,    0.034034,    0.035035,    0.036036,    0.037037,    0.038038,    0.039039,     0.04004,    0.041041,    0.042042,    0.043043,    0.044044,    0.045045,    0.046046,    0.047047,
          0.04804

In [10]:
metrics = model.val()  # 학습이 끝난 상태에서 실행
print(metrics)

Ultralytics 8.3.58 🚀 Python-3.10.15 torch-2.5.0 CUDA:0 (NVIDIA GeForce RTX 4090, 24564MiB)
Model summary (fused): 168 layers, 3,006,233 parameters, 0 gradients, 8.1 GFLOPs


val: Scanning /mnt/d/ships/datasets/labels/val.cache... 3844 images, 0 backgrounds, 0 corrupt: 100%|██████████| 3844/3844 [00:00<?, ?it/s]
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100%|██████████| 241/241 [00:15<00:00, 15.32it/s]


                   all       3844       4522      0.965      0.961      0.981      0.796
                  boat       3365       3642       0.98      0.993      0.994      0.846
                  buoy        716        880       0.95      0.928      0.967      0.746
Speed: 0.1ms preprocess, 0.5ms inference, 0.0ms loss, 0.5ms postprocess per image
Results saved to runs/detect/model2
ultralytics.utils.metrics.DetMetrics object with attributes:

ap_class_index: array([1, 2])
box: ultralytics.utils.metrics.Metric object
confusion_matrix: <ultralytics.utils.metrics.ConfusionMatrix object at 0x7fb53f0a58a0>
curves: ['Precision-Recall(B)', 'F1-Confidence(B)', 'Precision-Confidence(B)', 'Recall-Confidence(B)']
curves_results: [[array([          0,    0.001001,    0.002002,    0.003003,    0.004004,    0.005005,    0.006006,    0.007007,    0.008008,    0.009009,     0.01001,    0.011011,    0.012012,    0.013013,    0.014014,    0.015015,    0.016016,    0.017017,    0.018018,    0.019019,    

In [13]:
# 1) best.pt 모델 불러오기 (학습 단계에서 model이 이미 best.pt를 인식 중이면 이 단계 생략 가능)
inference_model = YOLO('/mnt/d/ships/model2/weights/best.pt')

# 2) 추론할 이미지(또는 폴더)
results = inference_model.predict(
    source='/mnt/d/ships/images.jpg',  # 또는 'test_images/*.jpg', 'test_video.mp4' 등
    conf=0.25,                # confidence threshold (기본값 0.25)
    save=True,                # 결과 이미지/영상 저장
    save_txt=True             # 라벨 정보(.txt) 저장
)


image 1/1 /mnt/d/ships/images.jpg: 640x480 (no detections), 144.0ms
Speed: 1.5ms preprocess, 144.0ms inference, 0.4ms postprocess per image at shape (1, 3, 640, 480)
Results saved to runs/detect/predict
0 label saved to runs/detect/predict/labels


In [17]:
import cv2
from ultralytics import YOLO

# 1) 학습 완료된 YOLO 모델 불러오기
model = YOLO('/mnt/d/ships/model2/weights/best.pt')  # best.pt 경로

# 2) 동영상 캡처 객체 생성 (파일 혹은 웹캠)
#    웹캠 실시간 테스트를 하려면 '0' 또는 다른 장치 인덱스를 넣을 수도 있음
cap = cv2.VideoCapture('/mnt/d/ships/test1.mp4')  

while True:
    ret, frame = cap.read()
    if not ret:
        break

    # 3) YOLO 모델로 현재 프레임 추론
    #    return_type='tensor'로 받아서 결과를 직접 그려도 되고,
    #    default로 받은 results에는 box, conf, cls 등의 정보가 들어있음
    results = model.predict(frame, conf=0.25, verbose=False)  # 원하는 conf threshold 설정

    # 4) 결과를 시각화(바운딩박스) 한 이미지 얻기
    #    Ultralytics의 'plot()' 함수는 자동으로 바운딩박스가 그려진 ndarray를 반환
    annotated_frame = results[0].plot()
    
    # 5) 화면에 표시
    cv2.imshow('YOLOv8 Detection', annotated_frame)

    # 'q' 누르면 중단
    if cv2.waitKey(1) & 0xFF == ord('q'):
        break

cap.release()
cv2.destroyAllWindows()